# Homework №5 | ODE-free Diffusion Distillation

In this homework, you will implement the [Latent Adversarial Diffusion Distillation](https://arxiv.org/abs/2403.12015) and [MMD Distillation](https://arxiv.org/pdf/2503.16397) approaches.

1) Latent Adversarial Diffusion Distillation **(7pt)**
2) MMD Distillation **(3pt)**

In [ ]:
!pip install diffusers==0.37.0 peft==0.18.0

In [ ]:
import csv
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from diffusers import StableDiffusionPipeline, UNet2DConditionModel, DDIMScheduler
from peft import LoraConfig, get_peft_model

from tqdm.auto import tqdm, trange

# Visualization
from IPython.display import clear_output
from PIL import Image

%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid")

torch.set_num_threads(16)

In [ ]:
#---------------------
# Visualization utils
#---------------------

def visualize_images(images):
    assert len(images) == 4
    plt.figure(figsize=(12, 3))
    for i, image in enumerate(images):
        plt.subplot(1, 4, i+1)
        plt.imshow(image)
        plt.axis('off')

    plt.subplots_adjust(wspace=-0.01, hspace=-0.01)


#--------------
# Tensor utils
#--------------

def extract_into_tensor(a, t, x_shape):
    b, *_ = t.shape
    out = a.gather(-1, t)
    return out.reshape(b, *((1,) * (len(x_shape) - 1)))

#---------------
# Dataset utils
#---------------

class TextImageDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, subset_name="train2014_5k", transform=None, max_cnt=None):
        """
        Arguments:
            root_dir (string): Директория с картинками
            transform (callable, optional): преобразования, применимые к картинкам
        """
        self.root_dir = root_dir
        self.transform = transform
        self.extensions = (
            ".jpg",
            ".jpeg",
            ".png",
            ".ppm",
            ".bmp",
            ".pgm",
            ".tif",
            ".tiff",
            ".webp",
        )
        sample_dir = os.path.join(root_dir, subset_name)

        # Collect image paths
        self.samples = sorted(
            [
                os.path.join(sample_dir, fname)
                for fname in os.listdir(sample_dir)
                if fname[-4:] in self.extensions
            ],
            key=lambda x: x.split("/")[-1].split(".")[0],
        )
        self.samples = (
            self.samples if max_cnt is None else self.samples[:max_cnt]
        )  # 

        # Load prompts
        self.captions = {}
        with open(
            os.path.join(root_dir, f"{subset_name}.csv"), newline="\n"
        ) as csvfile:
            spamreader = csv.reader(csvfile, delimiter=",")
            for i, row in enumerate(spamreader):
                if i == 0:
                    continue
                self.captions[row[1]] = row[2]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        sample_path = self.samples[idx]
        sample = Image.open(sample_path).convert("RGB")

        if self.transform:
            sample = self.transform(sample)

        return {
            "image": sample,
            "text": self.captions[os.path.basename(sample_path)],
            "idxs": idx, }

### [Stable Diffusion 1.5](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5)

**SD1.5** uses the $\epsilon$-parameterization and classical VP process and consists of the following components:

1) **VAE** — maps $3{\times}512{\times}512$ images into $4{\times}64{\times}64$ latents.
2) **Text encoder** — extracts text features from text prompts.
3) **Diffusion model** — a UNet that operates on “latent images” of size $4{\times}64{\times}64$.

In [ ]:
pipe = <YOUR CODE> # Same as in HW4

# Check that all model components are in FP16 and on cuda
assert pipe.unet.dtype == torch.float16 and pipe.unet.device.type == 'cuda'
assert pipe.vae.dtype == torch.float16 and pipe.vae.device.type == 'cuda'
assert pipe.text_encoder.dtype == torch.float16 and pipe.text_encoder.device.type == 'cuda'

# Replace the default solver with DDIM
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config, timestep_spacing="trailing")
pipe.scheduler.timesteps = pipe.scheduler.timesteps.cuda()
pipe.scheduler.alphas_cumprod = pipe.scheduler.alphas_cumprod.cuda()

# Define the teacher model that we plan to distill
teacher_unet = pipe.unet
teacher_unet.requires_grad_(False);

##  Creating the dataset

We will finetune the model on a small training set of 10,000 text-image pairs generated by the [FLUX](https://huggingface.co/black-forest-labs/FLUX.1-dev) model.

The data can be downloaded using the commands in the cell below. The current directory ./ should then contain:
* Folder flux_data with 10000 images
* File flux_data.csv with 10000 text prompts

The data is parsed correctly by the already implemented **TextImageDataset** class.

In [ ]:
!wget https://storage.yandexcloud.net/yandex-research/flux_data_10k.tar.gz
!tar -xzf flux_data_10k.tar.gz

In [ ]:
from torchvision import transforms

transform = transforms.Compose(
    [
        transforms.Resize(512),
        transforms.CenterCrop(512),
        transforms.ToTensor(),
        lambda x: 2 * x - 1,
    ]
)
dataset = TextImageDataset(".",
    subset_name="flux_data",
    transform=transform
)

batch_size = 8 # Recommended batch size for colab 

train_dataloader = torch.utils.data.DataLoader(
    dataset=dataset, shuffle=True, batch_size=batch_size, drop_last=True
)

In [ ]:
@torch.no_grad()
def prepare_batch(batch, pipe):
    """
    Preprocess a batch of textual prompts and images to corresponding embeddings,
    using the text encoder and VAE
    Params:
    
    Return:
        latents: torch.Tensor([B, 4, 64, 64], dtype=torch.float16)
        prompt_embeds: torch.Tensor([B, 77, D], dtype=torch.float16)
    """
    
    # Tokenize prompts
    text_inputs = pipe.tokenizer(
        batch['text'],
        padding="max_length",
        max_length=pipe.tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    )
    
    # Exctract prompt embeddings using the text encoder
    prompt_embeds = pipe.text_encoder(text_inputs.input_ids.cuda())[0]

    # Map images to the VAE latent space
    image = batch['image'].to("cuda", dtype=torch.float16)
    latents = pipe.vae.encode(image).latent_dist.sample()
    latents = latents * pipe.vae.config.scaling_factor
    return latents, prompt_embeds

## Latent Adversarial Diffusion Distillation (7 pt)

---

We train a 4-step GAN model initialized from the teacher U-Net.

The discriminator is also based on the teacher model, augmented with a small discriminator head applied to the teacher’s features.

Both the generator and discriminator operate in the VAE latent space (hence Latent ADD).

The diffusion-based discriminator enables feature extraction at multiple noise levels. Therefore, it operates on noisy images for $t \in [t_{min}, t_{max}]$.

More details in the particular cells below.

### Preparing the student model and optimizer

First, we create the trainable student model: a UNet initialized with the SD1.5 weights. You should use the `UNet2DConditionModel` class and load **only** the UNet component from SD1.5.

In [ ]:
unet = <YOUR CODE> # Same as in HW4

assert unet.dtype == torch.float32
assert unet.training

In [ ]:
# Setup the model layers to which LoRA is applied
lora_modules = [
    "to_q", "to_k", "to_v", "to_out.0", "proj_in", "proj_out",
    "ff.net.0.proj", "ff.net.2", "conv1", "conv2", "conv_shortcut",
    "downsamplers.0.conv", "upsamplers.0.conv", "time_emb_proj"
]

lora_config = LoraConfig(
    r=64, # Matrix rank for A and B in LoRA.
    lora_alpha=1, # controls a learning rate for LoRA training: lr_lora = lr_base * lora_alpha / r. If only LoRA parameters are being trained, you can simply adjust the learning rate in the optimizer.
    target_modules=lora_modules
)


# Create a wrapper around the original UNet model with LoRA adapters using the PEFT library
unet = get_peft_model(unet, lora_config, adapter_name="add")

# Create student optimizer
optimizer = torch.optim.AdamW(unet.parameters(), lr=2e-4)

# Enable gradient checkpointing - important technique for memory efficiency
unet.enable_gradient_checkpointing()
teacher_unet.enable_gradient_checkpointing()

### Generator prediction (1 pt)

In both LADD and MMD, we need to perform the generator prediction at each training iteration. Similar to classical Consistency Models, the generator is trained to predict $\mathbf{x}_0$ at each step.

1) Our generator follows the 4-step timestep schedule $t \in [249, 499, 749, 999]$. We train the generator to operate only on these timesteps for simplicity.

2) Given a batch of real image latents, we apply the forward diffusion process $q(\mathbf{x}_t | \mathbf{x}_0)$ to random $t \in [249, 499, 749, 999]$

3) The generator predicts $\epsilon$ for each $\mathbf{x}_t$ and converts them to $\mathbf{x}_0$.

Use `scheduler.add_noise()` to apply the forward process $q(\mathbf{x}_t | \mathbf{x}_0)$

In [ ]:
def generator_prediction(
    latents,
    prompt_embeds,
    unet, 
    scheduler,
    generator_timesteps=[249, 499, 749, 999],
):
    <YOUR CODE>
    return x0_pred

### Prepairing a discriminator (2 pt)

Discriminator operates on top of the `teacher_unet` features. 

Implement the Discriminator class:

* **init**: 
    1) Design and create the discriminator head. It can be either a small convolution network or MLP applied to the pooled features ("pooled" - averaged pooling over spatial dimensions of the feature map). **Note:** the head can be simple and its particular design does not matter much.
    2) Register the forward hook to extract the unet features from the middle block.
* **forward:**
    1) Extract teacher features for given inputs
    2) Compute logits using the discriminator head

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, teacher_unet, num_discriminator_layers=4):
        """
        Create the discriminator head and register the forward hook.
        """
        super().__init__()
        self.unet = teacher_unet
        
        # Create the discriminator head
        <YOUR CODE>
        
        # Register the forward hook
        <YOUR CODE>
        
    def forward(self, *args, **kwargs):
        """
        Extract features and return the discriminator logits
        
        Remember to convert half-precision features to FP32 
        """
        <YOUR CODE>
        return logits

#### Create the discriminator and optmizer

In [ ]:
D_unet = Discriminator(teacher_unet).cuda()
D_optimizer = torch.optim.AdamW(D_unet.parameters(), lr=2e-4)

### GAN loss (1 pt)

Implement the GAN loss on top of the teacher unet features.

We suggest implementing a classic non-saturating GAN objective:

**Discriminator loss** 

$\mathcal{L}_D=-\mathbb{E}_{\mathbf{x}\sim p_{\mathrm{data}}}\big[\log D(\mathbf{x})\big]
-\mathbb{E}_{\mathbf{z}\sim p(\mathbf{z})}\big[\log(1 - D(G(\mathbf{z})))\big]$

**Generator loss** 

$\mathcal{L}_G - \mathbb{E}_{\mathbf{z}\sim p(\mathbf{z})}\big[\log D(G(\mathbf{z}))\big]$

---

**Optional:** You may also try other GAN objectives, e.g., [WGAN](https://arxiv.org/abs/1701.07875).

**WGAN | Discriminator loss** 

$\mathcal{L}_D
=
\mathbb{E}_{\mathbf{z}\sim p(\mathbf{z})}\big[D(G(\mathbf{z}))\big]
-
\mathbb{E}_{\mathbf{x}\sim p_{\mathrm{data}}}\big[D(\mathbf{x})\big]$

**WGAN | Generator loss** 

$\mathcal{L}_G
=
-\mathbb{E}_{\mathbf{z}\sim p(\mathbf{z})}\big[D(G(\mathbf{z}))\big]$


In [ ]:
def gan_loss_fn(fake_logits, real_logits=None):
    if real_logits is not None:
        # Discriminator loss
        <YOUR CODE>
    else:
        # Generator loss
        <YOUR CODE>
    return loss

In [ ]:
def get_x0_from_noise(sample, model_output, scheduler, timestep):
    """ 
    epsilon-prediction --> x-prediction
    """
    
    <YOUR CODE>
    return x0_pred

### Implement discriminator loss calculation (2 pt)

In [ ]:
def discriminator_loss(
    real_latents,
    prompt_embeds, 
    unet, 
    D_unet,
    scheduler,
    t_min=0,
    t_max=200
):
    """ 
    Compute the discriminator (D) loss
    
    Steps:
    1) Perform the student prediction to obtain fake_latents
    2) Prepare the D input: sample timesteps in [t_min, t_max] and apply the forward process to both real and fake latents
    3) Compute real and fake logits and calculate the D loss
        
    Hint: pay attention which parameters are 'requires_grad=False' and use `torch.no_grad()' appropriately
    Recommendation: Consider torch.autocast within torch.no_grad() for efficiency 
    """
    <YOUR CODE>
    return D_loss 

### Implement generator loss calculation (1 pt)

In [ ]:
def generator_loss(
    real_latents,
    prompt_embeds, 
    unet,
    D_unet,
    scheduler,
    t_min=0,
    t_max=200
):
    """ 
    Compute the generator (G) loss
    
    Steps:
    1) Perform the student prediction to obtain fake_latents
    2) Prepare the D input: sample timesteps in [t_min, t_max] and apply the forward process to both real and fake latents
    3) Compute real and fake logits and calculate the D loss
    """
    <YOUR CODE>
    return G_loss

### Training loop

You are given a training loop that trains the model in full precision (FP32) with a batch size of 8. Similar to all previous HWs you can add `mixed-precision FP16/FP32`. **Note that mixed-precision training is only needed for the generator update**

#### Recommended setups

**Debug mode:** `discriminator_iters=1` and `generator_training_iters=100`. You are expected to see reasonable but  very blurry images at this stage.

**Final mode:**  `discriminator_iters=2` and `generator_training_iters=300`. If it takes too long in your environment, you can set `discriminator_iters=1`. 

**Optional mode:** if you want to see better results, increase `generator_training_iters` to 600.

In [ ]:
torch.cuda.empty_cache()

t_min = 0
t_max = 200

D_loss_history = []
G_loss_history = []

discriminator_iters = 2 # Number of discriminator updates per a single generator update 
generator_training_iters = 300 # Make sure that generator_training_iters * discriminator_iters * batch size < 10000
loader = iter(train_dataloader)

for i in trange(generator_training_iters):
    
    # Update the discriminator discriminator_iters times    
    for _ in range(discriminator_iters):
        
        # Sample batch for discriminator update
        batch = next(loader)
        latents, prompt_embeds = prepare_batch(batch, pipe)
        
        D_loss = discriminator_loss(
            latents, 
            prompt_embeds, 
            unet, 
            D_unet, 
            pipe.scheduler,
            t_min=t_min,
            t_max=t_max,
        )
        
        D_loss.backward()
        D_optimizer.step()
        D_optimizer.zero_grad(set_to_none=True)
        
    # Update the generator
    
    # Sample batch for generator update
    batch = next(loader)
    latents, prompt_embeds = prepare_batch(batch, pipe)
    
    G_loss = generator_loss(
        latents, 
        prompt_embeds, 
        unet, 
        D_unet, 
        pipe.scheduler,
        t_min=t_min,
        t_max=t_max,
    )
    
    # Update parameters
    G_loss.backward()
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    
    D_loss_history.append(D_loss.item())
    G_loss_history.append(G_loss.item())
    
    # Visualize losses
    if len(G_loss_history) % 20 == 0:
        clear_output(wait=True)
        
        plt.figure(figsize=(16, 6))
        plt.subplot(1, 2, 1)
        loss_values = np.array(G_loss_history)
        iters = np.arange(len(loss_values))
        
        window = 10
        smooth_loss = np.convolve(loss_values, np.ones(window)/window, mode='valid')

        plt.plot(iters, loss_values, color="#4C72B0", alpha=0.25, linewidth=1, label="Raw loss")
        plt.plot(iters[window-1:], smooth_loss, color="#4C72B0", linewidth=2.5, label="Smoothed loss")

        plt.title("Generator loss", fontsize=16)
        plt.xlabel("Train iterations", fontsize=12)
        plt.ylabel("Loss", fontsize=12)
        plt.legend(frameon=False, fontsize=12)
        
        plt.subplot(1, 2, 2)
        loss_values = np.array(D_loss_history)
        smooth_loss = np.convolve(loss_values, np.ones(window)/window, mode='valid')

        plt.plot(iters, loss_values, color="#4C72B0", alpha=0.25, linewidth=1, label="Raw loss")
        plt.plot(iters[window-1:], smooth_loss, color="#4C72B0", linewidth=2.5, label="Smoothed loss")

        plt.title("Discriminator loss", fontsize=16)
        plt.xlabel("Train iterations", fontsize=12)
        plt.ylabel("Loss", fontsize=12)
        plt.legend(frameon=False, fontsize=12)
        
        plt.tight_layout()
        plt.show()

### Reference loss dynamic

We provide the loss dynamics for `discriminator_iters=2` and `generator_training_iters=300`. Note that actual loss values can be different based on your implementation. 


**High-level loss behavior explanation** 

* At the beginning, the discriminator is warming up and gets stronger rapidly, making the generator task more challenging. (D_loss $\downarrow$, G_loss $\uparrow$) 

* Then, the generator gets stronger and starts fooling the discriminator more succesfully. (D_loss $\uparrow$, G_loss $\downarrow$)

<div>
<img src="https://storage.yandexcloud.net/yandex-research/visual-genai/hw5/reference_add_final_loss.jpg" width="1000"/>
</div>


### Multistep Stochastic Sampling (aka Consistency Sampling)

For generation, we need `multistep stochastic sampling` which you already implemented for Consistency Models in the previous HW.

<div>
<img src="https://storage.yandexcloud.net/yandex-research/cvweek-cd-task-images/cd-sampling-idea.jpg" width="600"/>
</div>

More formally:

$x_{t_n} \sim {N}(0, I)$

$for\ t_i \in [t_n, ..., t_1]:$

* $\epsilon \leftarrow unet(x_{t_i})$

* $x_0 \leftarrow \text{DDIM}(\epsilon, x_{t_i}, t_i, 0)$

* $x_{t_{i-1}} \leftarrow q(x_{t_{i-1}} | x_0)$


**You can use the consitency_sampling function from HW4 as is.**


In [ ]:
@torch.no_grad()
def multistep_sampling(
    pipe,
    prompt,
    num_inference_steps=4,
    generator=None,
    num_images_per_prompt=4,
):
    if prompt is not None and isinstance(prompt, str):
        batch_size = 1
    elif prompt is not None and isinstance(prompt, list):
        batch_size = len(prompt)

    device = pipe._execution_device

    # Extract prompt embeddings using pipe.encode_prompt
    prompt_embeds = <YOUR CODE HERE>
    assert prompt_embeds.dtype == torch.float16

    # Setup the few step scheduler
    assert pipe.scheduler.config['timestep_spacing'] == 'trailing'
    pipe.scheduler.set_timesteps(num_inference_steps)

    # Create a latent batch N(0,I) of the appropriate shape
    latents = <YOUR CODE HERE>

    # Sampling loop
    <YOUR CODE HERE>

    # Decode latents into pixels. Do not forget to rescale the latents using pipe.vae.config.scaling_factor 
    image = <YOUR CODE HERE>
    do_denormalize = [True] * image.shape[0]
    image = pipe.image_processor.postprocess(image, output_type="pil", do_denormalize=do_denormalize)
    return image

### Generate images 

In [ ]:
validation_prompts = [
    "A sad puppy with large eyes",
    "A girl with pale blue hair and a cami tank top",
    "A lighthouse in a giant wave, origami style",
    "belle epoque, christmas, red house in the forest, photo realistic, 8k",
    "A small cactus with a happy face in the Sahara desert",
]

In [ ]:
pipe.unet = unet.eval().to(torch.float16)
assert unet.active_adapter == 'add'

for prompt in validation_prompts:
    generator = torch.Generator(device="cuda").manual_seed(1)

    images = multistep_sampling(
        pipe,
        prompt,
        num_inference_steps=4,
        num_images_per_prompt=4,
        generator=generator,
    )

    visualize_images(images)

### References

**Important:** These results serve only as a rough quality reference. Your outputs may differ much, and that is expected. Likely, you will get much better images :)

![img](https://storage.yandexcloud.net/yandex-research/visual-genai/hw5/reference_add_final_image_dog.jpg)

![img](https://storage.yandexcloud.net/yandex-research/visual-genai/hw5/reference_add_final_image_girl.jpg)

![img](https://storage.yandexcloud.net/yandex-research/visual-genai/hw5/reference_add_final_image_lighthouse.jpg)

![img](https://storage.yandexcloud.net/yandex-research/visual-genai/hw5/reference_add_final_image_house.jpg)

![img](https://storage.yandexcloud.net/yandex-research/visual-genai/hw5/reference_add_final_image_cactus.jpg)

## Maximum Mean Discrepancy (MMD) Distillation (3 pt)


Implement the **patch-level** MMD distillation approach, which computes the MMD distance between distributions induced by **individual image pairs** $(\mathbf{x}_{real}, \mathbf{y}_{fake})$.

---

Here is the visualization for DiT-based architectures:

<div>
<img src="https://storage.yandexcloud.net/yandex-research/visual-genai/hw5/mmd_distill_scheme.jpg" width="800"/>
</div>

You need to implement this for the UNet backbone.

**Algorithm for a single image pair** $(\mathbf{x}_{real}, \mathbf{y}_{fake})$ 


1) Apply $q(\mathbf{x}_t | \mathbf{x})$ to both $\mathbf{x}_{real}$ and $\mathbf{y}_{fake}$ for $t \in [t_{min}, t_{max}]$

2) Extract intermediate feature maps from the teacher model $f_\psi$ : $f_\psi(\mathbf{x}_t, t) = F^{1{\times}C{\times}H{\times}W}$, where $C$ - hidden size, $H$ and $W$ - feature map spatial sizes.

3) Treat each spatial feature vector $F[0,:,i,j]$ as an individual patch-level representation $\to$ each image represents the distribution $p$ of patch-level representations. 

4) Having $p_{real}$ and $p_{fake}$ for real and fake images, respectively, we can compute $\mathrm{MMD}^2(p_{real},p_{fake})$.


$\mathrm{MMD}^2(p_{real},p_{fake})
=
\mathbb{E}_{\mathbf{x},\mathbf{x}'\sim p_{real}}[k(\mathbf{x},\mathbf{x}')]
+
\mathbb{E}_{\mathbf{y},\mathbf{y}'\sim p_{fake}}[k(\mathbf{y},\mathbf{y}')]
-
2\,\mathbb{E}_{\mathbf{x}\sim p_{real},\,\mathbf{y}\sim p_{fake}}[k(\mathbf{x},\mathbf{y})]$

**Note that $\mathbf{x}$ and $\mathbf{y}$ are spatial feature vectors $F[0,:,i,j]$, not images.**

---




In this task, you are asked to implement $\mathrm{MMD}^2$ for two kernels: `linear` and `rbf`

---

**Linear kernel**: $k(x_i, y_j) = x_i^\top y_j$

What is the resulting expression for $\mathrm{MMD}^2(p_{real},p_{fake})$ with the linear kernel? **< YOUR RESPONSE >**


---

**RBF kernel**: $k(x_i, x_j) = \exp\left(-\frac{\|x_i - x_j\|^2}{2\sigma^2}\right)$

To estimate MMD in this case, we will use the following unbiased estimation:

$\mathrm{MMD}^2
=
\frac{1}{n(n-1)} \sum_{i \ne j} k(x_i, x_j)
+
\frac{1}{m(m-1)} \sum_{i \ne j} k(y_i, y_j)
-
\frac{2}{nm} \sum_{i,j} k(x_i, y_j)$


P.S.: The biased estimation is also fine for n >> 1 and m >> 1

### Implement a feature extractor 

Similar to what you already did for the discriminator.

In [ ]:
def extract_teacher_features(teacher_unet, *args, **kwargs):
    """
        1) Extract teacher feature maps [B, C, H, W]
        2) Transform [B, C, H, W] feature maps into [B, H*W, C]
    """
    <YOUR CODE>
    return features

### MMD loss (2 pt)

Implement the patch-level MMD using the extracted features.

* Implement the **Linear (0.5 pt)** and **RBF (1.5 pt)** kernels as discussed above

**Note:** if you implemented both linear and RBF kernels, you can train the model only for the RBF one to save some time.

In [ ]:
def mmd_loss_fn(x, y, kernel="linear", sigma=200):
    """
    Patch-level MMD loss.
    
    Args:
        x [B, H*W, C]: fake features (H*W - feature map spatial size, C - feature map hidden size)
        y [B, H*W, C]: real features 
        kernel: MMD kernels ['linear', 'rbf']
        sigma: sigma parameter for the RBF kernel 
    """
    assert x.ndim == 3

    x = x.float()
    y = y.float()

    if kernel == "rbf":
        <YOUR CODE>
    elif kernel == "linear":
        <YOUR CODE>
    else:
        raise ValueError(f"Unsupported MMD kernel: {kernel}")

    return mmd.mean()

### Implement the generator loss for MMD distillation (1 pt)

In [ ]:
def mmd_generator_loss(
    real_latents,
    prompt_embeds, 
    unet,
    teacher_unet,
    scheduler,
    mmd_kernel="linear",
    t_min=0,
    t_max=200,
):
    """ 
    Compute the MMD distillation generator loss
    
        1) Predict fake latents with the generator
        2) Prepare the teacher_unet inputs: sample timesteps in [t_min, t_max] and apply the forward process to both real and fake latents
        3) Extract real and fake features and calculate the MMD loss
    """
    <YOUR CODE>
    return G_loss
    

In [ ]:
mmd_kernel = 'rbf' 

unet = unet.to(torch.float32)
unet.train()
assert unet.dtype == torch.float32

# Add new LoRA adapters for the MMD model
unet.add_adapter(f"mmd_{mmd_kernel}", lora_config)
unet.set_adapter(f"mmd_{mmd_kernel}")

# Create a new optimizer
optimizer = torch.optim.AdamW(unet.parameters(), lr=2e-4)

### Training loop

**Recommended final setup:** `generator_training_iters = 300` for both kernels

In [ ]:
torch.cuda.empty_cache()

t_min = 0
t_max = 200

loss_history = []

generator_training_iters = 300
loader = iter(train_dataloader)

for i in trange(generator_training_iters):
    
    batch = next(loader)
    latents, prompt_embeds = prepare_batch(batch, pipe)
    
    # Update generator
    loss = mmd_generator_loss(
        latents,
        prompt_embeds, 
        unet,
        teacher_unet,
        pipe.scheduler,
        mmd_kernel=mmd_kernel,
        t_min=0,
        t_max=200,
    )
    
    # Update params
    loss.backward()
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
        
    loss_history.append(loss.item())
    
    # Visualize loss
    if len(loss_history) % 20 == 0:
        clear_output(wait=True)
        
        loss_values = np.array(loss_history)
        iters = np.arange(len(loss_values))
        
        window = 10
        smooth_loss = np.convolve(loss_values, np.ones(window)/window, mode='valid')

        plt.figure(figsize=(8, 6))
        plt.plot(iters, loss_values, color="#4C72B0", alpha=0.25, linewidth=1, label="Raw loss")
        plt.plot(iters[window-1:], smooth_loss, color="#4C72B0", linewidth=2.5, label="Smoothed loss")

        plt.xlabel("Train iterations", fontsize=12)
        plt.ylabel("Loss", fontsize=12)
        plt.legend(frameon=False, fontsize=12)
        plt.tight_layout()
        plt.show()

In [ ]:
pipe.unet = unet.eval().to(torch.float16)
assert unet.active_adapter == f'mmd_{mmd_kernel}'

for prompt in validation_prompts:
    generator = torch.Generator(device="cuda").manual_seed(1)

    images = multistep_sampling(
        pipe,
        prompt,
        num_inference_steps=4,
        num_images_per_prompt=4,
        generator=generator,
    )

    visualize_images(images)


### Reference loss dynamics | Linear

<div>
<img src="https://storage.yandexcloud.net/yandex-research/visual-genai/hw5/reference_mmd_linear_loss.jpg" width="500"/>
</div>

### Reference loss dynamics | RBF

<div>
<img src="https://storage.yandexcloud.net/yandex-research/visual-genai/hw5/reference_mmd_rbf_kernel_loss.jpg" width="500"/>
</div>

### Reference images | MMD + RBF


![img](https://storage.yandexcloud.net/yandex-research/visual-genai/hw5/reference_mmd_rbf_image_dog.jpg)

![img](https://storage.yandexcloud.net/yandex-research/visual-genai/hw5/reference_mmd_rbf_image_girl.jpg)

![img](https://storage.yandexcloud.net/yandex-research/visual-genai/hw5/reference_mmd_rbf_image_lighthouse.jpg)

![img](https://storage.yandexcloud.net/yandex-research/visual-genai/hw5/reference_mmd_rbf_image_house.jpg)

![img](https://storage.yandexcloud.net/yandex-research/visual-genai/hw5/reference_mmd_rbf_image_cactus.jpg)